In [ ]:
%pip install --quiet s3fs
%pip install --quiet pandas
%pip install --quiet scikit-learn
%pip install --quiet seaborn
%pip install --quiet matplotlib
%pip install --quiet joblib
%pip install --quiet skl2onnx
%pip install --quiet onnx
%pip install --quiet onnxruntime

In [ ]:
import sys
sys.version

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import onnx
import os

from skl2onnx import convert_sklearn
from skl2onnx.common.data_types import FloatTensorType

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, confusion_matrix, classification_report, ConfusionMatrixDisplay
)
from sklearn.pipeline import Pipeline

sns.set(style="whitegrid", context="notebook")

In [ ]:
import s3fs
import pandas as pd

# ---------------------------------------------------------------------------
# MinIO connection — uses cluster-internal endpoint via s3fs
# ---------------------------------------------------------------------------
MINIO_ENDPOINT_URL    = "http://minio-api.mlops.svc.cluster.local:9000"
AWS_ACCESS_KEY_ID     = "minioadmin"
AWS_SECRET_ACCESS_KEY = "minioadminpassword"
AWS_REGION            = "us-east-1"
BUCKET_NAME           = "datasets"
FILE_KEY              = "synthetic_uk_attainment_10000_clean_1.csv"

fs = s3fs.S3FileSystem(
    key                  = AWS_ACCESS_KEY_ID,
    secret               = AWS_SECRET_ACCESS_KEY,
    endpoint_url         = MINIO_ENDPOINT_URL,
    client_kwargs        = {"region_name": AWS_REGION},
    config_kwargs        = {"signature_version": "s3v4"},
)

# Confirm bucket is accessible
buckets = fs.ls("/")
assert BUCKET_NAME in buckets, f"Bucket '{BUCKET_NAME}' not found. Available: {buckets}"
print(f"✓ Connected to MinIO. Bucket '{BUCKET_NAME}' confirmed.")

# Load CSV directly into DataFrame
s3_path = f"{BUCKET_NAME}/{FILE_KEY}"
with fs.open(s3_path, 'r') as f_obj:
    df = pd.read_csv(f_obj)

print(f"✓ DataFrame loaded.  Shape: {df.shape}")

In [ ]:
# ---------------------------------------------------------
# FIX 1 & 3: Single unified, contiguous ordinal grade mapping
# ---------------------------------------------------------
# All letter-grade columns use the same 0-6 scale.
# U=0, E=1, D=2, C=3, B=4, A=5, A*=6
# This gives 7 contiguous classes matching config.pbtxt dims: [7]

GRADE_ORDER = ["U", "E", "D", "C", "B", "A", "A*"]
GRADE_TO_NUM = {g: i for i, g in enumerate(GRADE_ORDER)}   # {'U':0, 'E':1, ..., 'A*':6}
NUM_TO_GRADE = {i: g for i, g in enumerate(GRADE_ORDER)}   # {0:'U', 1:'E', ..., 6:'A*'}

# AS-level uses the same scale minus A* (A is the top grade for AS)
AS_GRADE_ORDER = ["U", "E", "D", "C", "B", "A"]
AS_TO_NUM = {g: i for i, g in enumerate(AS_GRADE_ORDER)}   # {'U':0, ..., 'A':5}

# FIX 3: Define helper EARLY so all subsequent cells can use it
def num_to_grade(v, mapping=NUM_TO_GRADE):
    """Convert numeric grade back to letter label."""
    return mapping.get(v, None)

# FIX 2: Single definition of encode_grade_series
def encode_grade_series(s, mapping, col_name):
    """Map a grade string series to numeric using mapping; warn on unknowns."""
    unknown = set(s.unique()) - set(mapping.keys())
    if unknown:
        print(f"Warning: {col_name} has unknown grades: {unknown}")
    return s.map(mapping)

PASS_GRADES = {"C", "B", "A", "A*"}

print("Grade mapping (0–6):", GRADE_TO_NUM)

In [ ]:
# Dataset overview — df already loaded from MinIO in the cell above
print(df.head())
print()
df.info()
print()
print("Missing values:")
print(df.isna().sum())

In [ ]:
# Basic stats
df.describe(include="all")

In [ ]:
# Grade distributions
print(df["SATS_score"].describe())

print("\nGCSE_grade distribution:")
print(df["GCSE_grade"].value_counts().sort_index())

print("\nGCE_AS_grade distribution:")
print(df["GCE_AS_grade"].value_counts().sort_index())

print("\nAlevel_Maths_grade distribution:")
print(df["Alevel_Maths_grade"].value_counts().sort_index())

In [ ]:
# ---------------------------------------------------------
# Validate required columns
# ---------------------------------------------------------
expected_cols = ["ref_id", "SATS_score", "GCSE_grade", "GCE_AS_grade", "Alevel_Maths_grade"]
missing = [c for c in expected_cols if c not in df.columns]
if missing:
    raise KeyError(f"Dataframe is missing required columns: {missing}")

# Working copy — FIX 6: keep ref_id
data = df[["ref_id", "SATS_score", "GCSE_grade", "GCE_AS_grade", "Alevel_Maths_grade"]].copy()

# Normalise string grade columns
data["GCE_AS_grade"]       = data["GCE_AS_grade"].astype(str).str.strip().str.upper()
data["Alevel_Maths_grade"] = data["Alevel_Maths_grade"].astype(str).str.strip().str.upper()

# Encode — FIX 1: using unified contiguous 0-6 scale
data["GCSE_grade_num"]         = data["GCSE_grade"].astype(int)
data["GCE_AS_grade_num"]       = encode_grade_series(data["GCE_AS_grade"], AS_TO_NUM, "GCE_AS_grade")
data["Alevel_Maths_grade_num"] = encode_grade_series(data["Alevel_Maths_grade"], GRADE_TO_NUM, "Alevel_Maths_grade")

# Verify
print(data[["ref_id", "GCSE_grade", "GCSE_grade_num",
            "GCE_AS_grade", "GCE_AS_grade_num",
            "Alevel_Maths_grade", "Alevel_Maths_grade_num"]].head(10))
print("\nUnique target classes:", sorted(data["Alevel_Maths_grade_num"].unique()))
# Expected: [0, 1, 2, 3, 4, 5, 6]  (7 classes)

In [ ]:
# EDA pairplot
eda_cols = ["SATS_score", "GCSE_grade_num", "GCE_AS_grade_num", "Alevel_Maths_grade_num"]

sns.pairplot(data[eda_cols], diag_kind="kde")
plt.suptitle("Pairplot of attainment variables", y=1.02)
plt.show()

In [ ]:
# Train–test split with stratification
feature_cols = ["SATS_score", "GCSE_grade_num", "GCE_AS_grade_num"]
target_col   = "Alevel_Maths_grade_num"

X = data[feature_cols]
y = data[target_col]

# FIX 6: preserve ref_id alignment
ref_ids = data["ref_id"]

X_train, X_test, y_train, y_test, ids_train, ids_test = train_test_split(
    X, y, ref_ids,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

print("Train shape:", X_train.shape, "  Test shape:", X_test.shape)
print("Train target distribution:\n", y_train.value_counts(normalize=True).sort_index())
print("Test  target distribution:\n", y_test.value_counts(normalize=True).sort_index())

In [ ]:
# Build and train Logistic Regression pipeline
# Note: `multi_class` param was removed in scikit-learn 1.6+.
# lbfgs handles multiclass natively — no argument needed.
log_reg_clf = Pipeline(
    steps=[
        ("scaler", StandardScaler()),
        (
            "logreg",
            LogisticRegression(
                solver="lbfgs",
                max_iter=2000,
                random_state=42,
            ),
        ),
    ]
)

log_reg_clf.fit(X_train, y_train)
print("Model trained. Classes:", log_reg_clf.classes_)
# Expected: [0 1 2 3 4 5 6]

In [ ]:
# Evaluate the model
y_pred = log_reg_clf.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification report (numeric grades):")

# FIX: build labels from the union of what's actually in y_test and y_pred.
# Using log_reg_clf.classes_ is unsafe — it includes all training classes,
# but a rare grade (e.g. U or A*) may be absent from the test split,
# causing a mismatch between sklearn's auto-detected class count and target_names.
actual_labels = sorted(np.unique(np.concatenate([y_test.values, y_pred])))
actual_names  = [NUM_TO_GRADE[i] for i in actual_labels]

print(classification_report(
    y_test, y_pred,
    labels=actual_labels,
    target_names=actual_names,
))

In [ ]:
# Confusion matrix — multi-class (numeric labels → letter tick labels)
present_classes = np.unique(np.concatenate([y_test, y_pred]))
label_names = [NUM_TO_GRADE[c] for c in present_classes]

cm = confusion_matrix(y_test, y_pred, labels=present_classes)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=label_names)
disp.plot(cmap="Blues", xticks_rotation=45)
plt.title("Confusion matrix — A-level Maths grade prediction")
plt.tight_layout()
plt.show()

In [ ]:
# FIX 7: Single pass/fail confusion matrix cell (duplicate removed)
y_test_labels  = pd.Series(y_test).map(num_to_grade)
y_pred_labels  = pd.Series(y_pred).map(num_to_grade)

y_test_binary  = y_test_labels.isin(PASS_GRADES).astype(int)
y_pred_binary  = y_pred_labels.isin(PASS_GRADES).astype(int)

cm_binary = confusion_matrix(y_test_binary, y_pred_binary)

plt.figure(figsize=(4, 3))
sns.heatmap(
    cm_binary, annot=True, fmt='d', cmap='Blues',
    xticklabels=['Pred Fail', 'Pred Pass'],
    yticklabels=['True Fail', 'True Pass']
)
plt.ylabel('True')
plt.xlabel('Predicted')
plt.title('Pass/Fail — Logistic Regression (Maths)')
plt.tight_layout()
plt.show()

In [ ]:
# FIX 6: Predictions with ref_id tracking
results_df = pd.DataFrame({
    "ref_id":           ids_test.values,
    "SATS_score":       X_test["SATS_score"].values,
    "GCSE_grade_num":   X_test["GCSE_grade_num"].values,
    "GCE_AS_grade_num": X_test["GCE_AS_grade_num"].values,
    "true_grade":       pd.Series(y_test.values).map(num_to_grade),
    "pred_grade":       pd.Series(y_pred).map(num_to_grade),
})

print(results_df.head(10))

In [ ]:
# Persist the trained sklearn model
model_path = "model.pkl"
joblib.dump(log_reg_clf, model_path)
print(f"Model saved to {model_path}")

In [ ]:
# Load and predict — deployment-style usage with ref_id
loaded_model = joblib.load(model_path)

new_students = pd.DataFrame({
    "ref_id":           [1001, 1002],
    "SATS_score":       [105,  115],
    "GCSE_grade_num":   [6,    8],
    "GCE_AS_grade_num": [4,    5],  # B, A
})

features_only = new_students[["SATS_score", "GCSE_grade_num", "GCE_AS_grade_num"]]
pred_nums     = loaded_model.predict(features_only)
pred_grades   = pd.Series(pred_nums).map(num_to_grade)

new_students["predicted_grade"] = pred_grades.values
print(new_students)

In [ ]:
# Convert sklearn pipeline to ONNX
# config.pbtxt expects:
#   input  name='input'        dims=[-1, 3]
#   output name='output_label' dims=[-1]

initial_type = [('input', FloatTensorType([None, 3]))]
options = {id(log_reg_clf): {"zipmap": False}}  # return plain array, not dict

onnx_model = convert_sklearn(
    log_reg_clf,
    initial_types=initial_type,
    options=options,
)

# Debug: show what skl2onnx actually named the outputs
print("Raw outputs from skl2onnx:")
for out in onnx_model.graph.output:
    print(f"  name='{out.name}'")

# FIX: only rename 'label' -> 'output_label'.
# Do NOT rename 'variable' or 'probabilities' — renaming multiple tensors
# to the same name breaks ONNX SSA (single static assignment) rules.
def rename_tensor(model, old_name, new_name):
    """Rename a tensor consistently across graph outputs and all node I/O."""
    for out in model.graph.output:
        if out.name == old_name:
            out.name = new_name
    for node in model.graph.node:
        for i, o in enumerate(node.output):
            if o == old_name:
                node.output[i] = new_name
        for i, inp in enumerate(node.input):
            if inp == old_name:
                node.input[i] = new_name

rename_tensor(onnx_model, "label", "output_label")
# Note: 'variable' (probabilities) is intentionally left as-is

print("\nONNX model outputs after rename:")
for out in onnx_model.graph.output:
    shape = [d.dim_value for d in out.type.tensor_type.shape.dim]
    print(f"  name='{out.name}'  shape={shape}")

In [ ]:
# Save the ONNX model
onnx_path = "model.onnx"

with open(onnx_path, "wb") as f:
    f.write(onnx_model.SerializeToString())

print("ONNX model saved to:", onnx_path)

# Validate the saved ONNX model
loaded_onnx = onnx.load(onnx_path)
onnx.checker.check_model(loaded_onnx)
print("ONNX model validation passed.")

In [ ]:
# Quick ONNX runtime inference check
# %pip install onnxruntime
import onnxruntime as rt
import numpy as np

sess = rt.InferenceSession(onnx_path)

# Test input matching config.pbtxt: input name='input', dims=[-1, 3]
test_input = np.array([[105, 6, 4]], dtype=np.float32)  # shape [1, 3]
onnx_pred  = sess.run(["output_label"], {"input": test_input})

print("ONNX prediction (raw):", onnx_pred)
print("Predicted class (numeric):", onnx_pred[0][0])
print("Predicted grade:", num_to_grade(int(onnx_pred[0][0])))

In [ ]:
# ---------------------------------------------------------------------------
# Generate Triton Inference Server config.pbtxt
# ---------------------------------------------------------------------------
config_pbtxt = """name: "mathmodel"
backend: "onnxruntime"
max_batch_size: 0

input [
  {
    name: "input"
    data_type: TYPE_FP32
    dims: [ -1, 3 ]
  }
]

output [
  {
    name: "output_label"
    data_type: TYPE_INT64
    dims: [ -1 ]
  }
]

instance_group [
  {
    count: 1
    kind: KIND_GPU
    gpus: [0]
  }
]
"""

config_path = "config.pbtxt"
with open(config_path, "w") as f:
    f.write(config_pbtxt)

print(f"✓ config.pbtxt written locally to: {config_path}")
print("\nContent preview:")
print(config_pbtxt)


In [ ]:
# ---------------------------------------------------------------------------
# Upload model artefacts to MinIO triton-model-repo bucket
# ---------------------------------------------------------------------------
# Target paths:
#   config.pbtxt  → triton-model-repo/models/mathmodel/config.pbtxt
#   *.onnx        → triton-model-repo/models/mathmodel/1/uk_attainment_logreg_math.onnx
# ---------------------------------------------------------------------------

TRITON_BUCKET   = "triton-model-repo"
MODEL_BASE_PATH = f"{TRITON_BUCKET}/models/mathmodel"

upload_map = {
    config_path: f"{MODEL_BASE_PATH}/config.pbtxt",
    onnx_path:   f"{MODEL_BASE_PATH}/1/model.onnx",
}

# Ensure the triton-model-repo bucket exists; create it if not
#if TRITON_BUCKET not in fs.ls("/"):
if not fs.exists(TRITON_BUCKET):
    fs.mkdir(TRITON_BUCKET)
    print(f"✓ Created bucket: {TRITON_BUCKET}")
else:
    print(f"✓ Bucket already exists: {TRITON_BUCKET}")

# Upload each file
for local_file, s3_dest in upload_map.items():
    print(f"\nUploading {local_file}  →  s3://{s3_dest} ...")
    fs.put(local_file, s3_dest)
    size = fs.info(s3_dest)["size"]
    print(f"  ✓ Uploaded  ({size:,} bytes)")

print("\n✓ All artefacts uploaded to MinIO successfully.")

# Verify by listing the model directory
print(f"\nContents of s3://{MODEL_BASE_PATH}:")
for entry in fs.find(MODEL_BASE_PATH):
    size = fs.info(entry)["size"]
    print(f"  {entry}  ({size:,} bytes)")